# Demo: Transforming Time-Series Data

Most forecasting models assume the data is *stationary* -- meaning the statistical properties don't change over time. Real data almost never starts that way. This demo shows how to check for stationarity and what to do when it fails.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [ ]:
# Load and clean the bike-share data (cleaning was covered in the previous module)
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
df = df.groupby("date", as_index=False)["rides"].mean()
df["date"] = pd.to_datetime(df["date"])
mask = (df["date"].dt.year == 2023) & (df["date"].dt.month == 10) & (df["rides"] < 1000)
df.loc[mask, "rides"] = df.loc[mask, "rides"] * 24
df.loc[df["rides"] < 0, "rides"] = np.nan
df = df.set_index("date").asfreq("D")
df["rides"] = df["rides"].interpolate(method="time")
rides = df["rides"]
print(f"Clean series: {len(rides)} days")

## Rolling Statistics

The simplest way to eyeball stationarity: compute a rolling mean and rolling standard deviation. If the mean drifts or the std changes shape over time, the series isn't stationary.

In [ ]:
window = 30
rmean = rides.rolling(window).mean()
rstd = rides.rolling(window).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color="lightgray", linewidth=0.5)
axes[0].plot(rmean.index, rmean.values, color="black", label="30-day mean")
axes[0].set_ylabel("Rides")
axes[0].legend()
axes[0].set_title("Rolling Mean")
axes[1].plot(rstd.index, rstd.values, color="tab:orange")
axes[1].set_ylabel("Rides")
axes[1].set_title("Rolling Std")
plt.tight_layout()
plt.show()

The rolling mean has a clear seasonal wave -- high in summer, low in winter. That's not stationary. The mean is changing over time.

## The ADF Test

Eyeballing is useful but not rigorous. The Augmented Dickey-Fuller test gives you a p-value. Below 0.05 means stationary. Above 0.05 means not stationary.

In [ ]:
result = adfuller(rides.dropna(), autolag="AIC")
print(f"ADF statistic: {result[0]:.4f}")
print(f"p-value:       {result[1]:.6f}")
print(f"Stationary:    {'yes' if result[1] < 0.05 else 'no'}")

## Differencing

When a series isn't stationary, differencing is the standard fix. `.diff()` replaces each value with the change from the previous day. This removes the level and often removes the trend too.

In [ ]:
diffed = rides.diff().dropna()

result_diff = adfuller(diffed, autolag="AIC")
print(f"Differenced ADF p-value: {result_diff[1]:.6f}")
print(f"Stationary: {'yes' if result_diff[1] < 0.05 else 'no'}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color="black", linewidth=0.5)
axes[0].set_ylabel("Rides")
axes[0].set_title("Original")
axes[1].plot(diffed.index, diffed.values, color="tab:green", linewidth=0.5)
axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_ylabel("Daily change")
axes[1].set_title("First-differenced")
plt.tight_layout()
plt.show()

The differenced series oscillates around zero with no visible trend. The ADF test confirms it's stationary. This is the version of the data that ARIMA-style models work with internally -- understanding this transformation is the whole point of this module.

## Train/Test Split

One last thing: when you split time-series data, you always split chronologically. Never randomly. The future can't leak into the training set.

In [ ]:
test_days = 90
train, test = rides.iloc[:-test_days], rides.iloc[-test_days:]
print(f"Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")